In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import os 
import plotly.express as px

In [61]:
max_rank = sche_unique.groupby('bill_code')['rank'].transform(max) 
sche_unique['rank_re'] = max_rank - sche_unique['rank'] 

C:\Users\VIET ANH\AppData\Local\Temp\ipykernel_18884\1415269964.py:1: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  max_rank = sche_unique.groupby('bill_code')['rank'].transform(max)


In [62]:
wh_1a = pd.read_csv('warehouse_1a.csv') 
set_1a = set(wh_1a['name']) 
sche_1a = sche_unique[sche_unique['warehouse_name'].isin(set_1a)].copy()
count_1a = sche_1a.groupby('bill_code').size().reset_index(name = 'count_1a') 
count_1a = count_1a[count_1a['count_1a'] >= 2] 
sche_unique = sche_unique[sche_unique['bill_code'].isin(set(count_1a['bill_code']))]


In [63]:
traces = pd.read_csv('output_all_traces/bill_trunk_traces.csv')

In [ ]:
traces_1a = traces[traces['bill_code'].isin(set(sche_unique['bill_code']))] 
traces_1a = traces_1a[~traces_1a['first_trunk'].isin(set_1a)] 
traces_1a = traces_1a[~traces_1a['last_trunk'].isin(set_1a)] 
traces_1a.to_csv(os.path.join('output_all_traces', 'traces_1a.csv'), index = False)

In [ ]:
origin = pd.read_csv("output_all_traces/origin_to_1A.csv") 
des = pd.read_csv('output_all_traces/1A_to_destination.csv') 

In [ ]:

print(traces_1a.shape) 
print(origin.shape)
print(des.shape)
print(sche_unique['bill_code'].nunique())

(499915, 7)
(737812, 7)
(673232, 7)
1039840


In [ ]:
sche_final = sche_unique[sche_unique['bill_code'].isin(set(traces_1a['bill_code']))].copy()

In [ ]:
def make_data(sche, title): 
    sche_1a_filter = sche[sche['warehouse_name'].isin(set_1a)]
    first = sche_1a_filter.groupby('bill_code')['rank'].min()
    last = sche_1a_filter.groupby('bill_code')['rank_re'].min()
    rank_first = first.value_counts().sort_index().reset_index()
    rank_last = last.value_counts().sort_index().reset_index()
    rank_first.columns = ['rank_to_1A', 'so_luong_bill']
    rank_last.columns = ['rank_1A_to', 'so_luong_bill']
    print(title)
    print(rank_first)
    print(rank_last)
    print()
    return rank_first, rank_last


through_least_2_first, through_least_2_last = make_data(sche_unique, "through at least two 1A")
traces_least_2_first, traces_least_2_last = make_data(sche_final, "traces well") 


through at least two 1A
   rank_to_1A  so_luong_bill
0           0         309935
1           1         701435
2           2          28300
3           3            158
4           4              8
5           5              2
6           6              2
    rank_1A_to  so_luong_bill
0            0         330194
1            1         643280
2            2          62873
3            3           2461
4            4            848
5            5            116
6            6             45
7            7             12
8            8              8
9            9              1
10          12              2

traces well
   rank_to_1A  so_luong_bill
0           1         480147
1           2          19653
2           3            104
3           4              7
4           5              2
5           6              2
   rank_1A_to  so_luong_bill
0           1         452841
1           2          44615
2           3           1762
3           4            575
4           5          

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6)) 
sns.barplot(data = traces_least_2_last, x="rank_1A_to", y = 'so_luong_bill')
plt.title('Thống kê số lượng Bill theo Rank của Kho 1A đầu tiên (1A -> destination)', fontsize=14, fontweight='bold')
plt.xlabel('Rank của Kho 1A đầu tiên', fontsize=12)
plt.ylabel('Số lượng Bill', fontsize=12)
for index, row in traces_least_2_last.iterrows(): 
    plt.text(index, row['so_luong_bill'], f"{int(row['so_luong_bill']):,}", 
    color = 'black', ha = 'center', va = 'bottom')
plt.tight_layout()
plt.savefig(os.path.join('output_plot', 'rank_to_1A.png'), bbox_inches = 'tight') 
plt.show()

In [ ]:
a = pd.read_csv('output_all_traces/origin_to_1A_filter.csv')


In [ ]:
a = a.sort_values('actual_weight', ascending=False)
print(a[['bill_code', 'actual_weight', 'time_1a_out']][:10])

In [ ]:
df = a.copy() 
df['time_1a_out'] = pd.to_datetime(df['time_1a_out']) 
df['out_15min'] = df['time_1a_out'].dt.floor('15min') 
trip_df = df.groupby(['kho_o1a', 'out_15min']).agg(
    total_bill = ('bill_code', 'nunique'),
    total_weight = ('actual_weight', 'sum'), 
    avg_wait_time = ('time_in_1a', 'mean')
).reset_index() 


trip_df['avg_wait_time'] = trip_df['avg_wait_time'].round(2)
trip_df['total_weight'] = trip_df['total_weight'].round(2)
trip_df['hour_out'] = trip_df['out_15min'].dt.hour + trip_df['out_15min'].dt.minute / 60.0 
trip_df = trip_df.sort_values(['total_weight', 'total_bill'], ascending=[False,False])
print(trip_df.shape)
trip_df.to_csv(os.path.join("output_all_traces", 'trip_1A.csv'), index = False) 

(18895, 6)


In [18]:
import json 
with open("D:\\optima\\VietAnh\\normal_bill_code_sample.json", 'r', encoding='utf-8') as f:
    data = json.load(f) 

head_data = data.get('head', []) 
tail_data = data.get('tail', []) 
head = pd.DataFrame(data['head'], columns = ['bill_code']) 
tail = pd.DataFrame(data['tail'], columns = ['bill_code'] )

In [19]:
origin = pd.read_csv('output_all_traces/origin_to_1A.csv') 
des = pd.read_csv('output_all_traces/1A_to_destination.csv')

In [20]:
print(origin.shape, des.shape)
print(head.shape, tail.shape)

(708217, 9) (640865, 9)
(1050594, 1) (1382029, 1)


In [21]:
warehouse = pd.read_csv('warehouse.csv') 
set_bc = warehouse[warehouse['Bưu Cục'] == "Y"]['name'].to_list()
print(set_bc)

['Kho Cao Bằng', 'Kho Trung Chuyển Cao Bằng', 'Kho Lạng Sơn', 'Kho Trung Chuyển Lạng Sơn', 'Kho Hạ Long-Quảng Ninh', 'Kho Uông Bí-Quảng Ninh', 'Kho Đông Triều-Quảng Ninh', 'Kho Cẩm Phả-Quảng Ninh', 'Kho Móng Cái-Quảng Ninh', 'Kho Quảng Yên', 'Kho Tổng Quảng Ninh', 'Kho Bưu cục Ngô Quyền', 'Kho An Dương', 'Kho Herbalife Hải Phòng ', 'Kho Kiến An', 'Kho Bưu cục Vĩnh Bảo', 'Kho Thái Bình', 'Kho Hưng Hà ', 'Kho Đông Hưng', 'Kho Tiền Hải', 'Kho Trung Chuyển Thái Bình', 'Kho Nam Định', 'Kho Hải Hậu', 'Kho Trực Ninh', 'Kho Trung Chuyển Nam Định', 'Kho Việt Trì', 'Kho Herbalife Phú Thọ', 'Kho Phú Thọ', 'Kho Bưu Cục Cẩm Khê', 'Kho Trung Chuyển Phú Thọ', 'Kho Thái Nguyên', 'Kho Bưu Cục Phổ Yên', 'Kho Trung Chuyển Thái Nguyên', 'Kho Yên Bái', 'Kho Nghĩa Lộ', 'Kho Bưu cục Nghĩa Lộ', 'Kho Trung Chuyển Yên Bái (Lào Cai)', 'Kho Tuyên Quang', 'Kho Trung Chuyển Tuyên Quang', 'Kho Hà Giang', 'Kho Trung Chuyển Hà Giang - Tuyên Quang', 'Kho Lào Cai', 'Kho Trung Chuyển Lào Cai', 'Kho Lai Châu', 'Kho Trung 

In [22]:
origin_bc = origin[origin['kho_o'].isin(set_bc)] 
des_bc = des[des['kho_d'].isin(set_bc)] 
print(origin_bc.shape, des_bc.shape)

(424866, 9) (524208, 9)


In [23]:
oh = origin_bc[origin_bc['bill_code'].isin(set(head['bill_code']))]
ho = head[~head['bill_code'].isin(set(origin_bc['bill_code']))]
dt = des_bc[des_bc['bill_code'].isin(set(tail['bill_code']))] 
td = tail[tail['bill_code'].isin(set(des_bc['bill_code']))]
print(oh.shape, ho.shape) 
print(dt.shape, td.shape)

(424858, 9) (625370, 1)
(524126, 9) (524600, 1)


In [24]:
oh = oh[((oh['status_o'] == 'OUT') & (oh['status_o1a'] == 'IN'))] 
dt = dt[((dt['status_d'] == 'IN') & (dt['status_d1a'] == 'OUT'))] 
print(oh.shape)
print(dt.shape) 
print(oh.head(2)) 
print(dt.head(2))

(376570, 9)
(414619, 9)
   bill_code            kho_o               time_o status_o        kho_o1a  \
0  B_0000001  Kho Nam Từ Liêm  2025-12-31 18:56:09      OUT  Kho Văn Giang   
1  B_0000002    Kho Chương Mỹ  2026-01-03 16:26:44      OUT  Kho Văn Giang   

              time_o1a status_o1a  actual_weight      time  
0  2026-01-01 03:33:07         IN           31.0  8.616111  
1  2026-01-03 21:47:48         IN           70.0  5.351111  
    bill_code        kho_d1a             time_d1a status_d1a  actual_weight  \
2   B_0000002  Kho Văn Giang  2026-02-28 01:31:01        OUT           70.0   
12  B_0000020  Kho Văn Giang  2026-01-26 09:11:44        OUT           90.0   

            kho_d               time_d status_d        time  
2   Kho Chương Mỹ  2026-02-28 07:53:15       IN    6.370556  
12  Kho Hải Dương  2026-02-03 10:33:18       IN  193.359444  


In [25]:
oh.to_csv(os.path.join('output_all_traces', 'origin_head.csv'), index = False) 
dt.to_csv(os.path.join('output_all_traces', 'destination_tail.csv'), index = False) 


In [26]:
print(oh.shape[0] / origin_bc.shape[0] * 100) 
print(dt.shape[0] / des_bc.shape[0] * 100)

88.63265123591908
79.09436712144797


In [17]:
print(origin_bc[~origin_bc['bill_code'].isin(oh['bill_code'].unique())])

        bill_code                  kho_o               time_o status_o  \
9       B_0000019          Kho Chương Mỹ  2026-01-21 16:18:21      OUT   
13      B_0000023        Kho Nam Từ Liêm  2026-01-26 18:39:45      OUT   
19      B_0000031          Kho Long Biên  2026-01-29 21:48:53      OUT   
44      B_0000060          Kho Yên Nghĩa  2026-02-05 19:58:49      OUT   
47      B_0000065       Kho BC Hoàng Mai  2026-02-07 10:45:20      OUT   
...           ...                    ...                  ...      ...   
708148  B_2547451         Kho Trung Tâm   2026-05-28 18:02:45      OUT   
708166  B_2547484     Kho Bưu Cục Ga Huế  2026-05-28 17:33:52      OUT   
708169  B_2547498         Kho Thường Tín  2026-05-28 17:43:12      OUT   
708181  B_2547533  Kho Bưu cục Ngô Quyền  2026-05-28 17:43:22      OUT   
708188  B_2547554       Kho BC Hoàng Mai  2026-05-28 17:48:16      OUT   

              kho_o1a             time_o1a status_o1a  actual_weight  time  
9       Kho Văn Giang  2026-01-22 

In [ ]:
post = pd.read_csv('post_office.csv') 
oh = pd.read_csv('output_all_traces/origin_head.csv') 
dt = pd.read_csv('output_all_traces/destination_tail.csv')

In [ ]:
warehouse = pd.read_csv('warehouse.csv') 
set_bc = warehouse[warehouse['Bưu Cục'] == "Y"]
print(set_bc)

In [ ]:
oh_post = oh[oh['kho_o'].isin(set(post['name']))] 
df_post = dt[dt['kho_d'].isin(set(post['name']))] 
print(oh_post.shape) 
print(dt.shape)

(9410, 7)
(517387, 7)


In [ ]:
warehouse = pd.read_csv('warehouse.csv')

In [ ]:
bc = post[post['name'].isin(set(warehouse['name']))]
print(bc.shape)

(14, 1)


Check bưu cục 






In [61]:
def build_data(df, col_a, col_b, origin): 
    df = df.copy() 
    df['pair'] = df[col_a].astype(str) + '->' + df[col_b].astype(str) 
    check_col = col_b if origin == False else col_a 
    agg = (df.groupby(['pair', check_col])
            .agg(
                so_bill = ('bill_code', 'nunique'), 
                tong_kg = ('actual_weight', 'sum')
            ).reset_index() )
    top = agg.nlargest(30, 'so_bill')
    result = top[check_col].to_list() 
    return result 


In [14]:
df_schedule = pd.read_csv('bill_schedule.csv', usecols=['bill_code', 'io_time', 'warehouse_name'])
df_bill = pd.read_csv('bill.csv', usecols=['bill_code', 'VD_type'])
post = pd.read_csv('post_office.csv') 
oh = pd.read_csv('output_all_traces/origin_head.csv') 
dt = pd.read_csv('output_all_traces/destination_tail.csv')

In [53]:
sche = df_schedule.copy() 

In [62]:
oh_list = build_data(oh, 'kho_o', 'kho_o1a', True) 
dt_list = build_data(dt, 'kho_d1a', 'kho_d', False) 

In [63]:

df_schedule['io_time'] = pd.to_datetime(df_schedule['io_time'])
df_schedule = df_schedule.sort_values(by=['bill_code', 'io_time'])

first_warehouses = df_schedule.drop_duplicates(subset=['bill_code'], keep='first')
target_warehouses = dt_list

filtered_bills = first_warehouses[first_warehouses['warehouse_name'].isin(target_warehouses)]

df_bill['VD_type'] = df_bill['VD_type'].astype(str).str.strip().str.upper()

merged_df = pd.merge(filtered_bills, df_bill, on='bill_code', how='inner')

results = []

for warehouse_name, group in merged_df.groupby('warehouse_name'):
    total_bills = len(group)
    
    if total_bills == 0:
        continue
    web_online_count = (group['VD_type'] == 'BILLWEBONLINE').sum()
    
    percentage = (web_online_count / total_bills) * 100

    # is_passed = percentage > x_percent
    
    # print(f"Kho: {warehouse_name}")
    # print(f"  - Tổng số bill: {total_bills}")
    # print(f"  - Số bill BILLWEBONLINE: {web_online_count}")
    # print(f"  - Tỉ lệ: {percentage:.2f}%")
    results.append([warehouse_name, total_bills, web_online_count, percentage])
    
    # results.append({
    #     'warehouse_name': warehouse_name,
    #     'total_bills': total_bills,
    #     'web_online_bills': web_online_count,
    #     'percentage': percentage
    # })



In [65]:
ans = [] 
for item in results:
    if item[3] <= 30:
        print(item)

['Bưu Cục 82 Chương Dương', 5483, np.int64(1471), np.float64(26.828378624840415)]
['Kho Bưu Cục Quận 9', 33156, np.int64(8119), np.float64(24.48727228857522)]
['Kho Bưu Cục Tam Kỳ', 5164, np.int64(1025), np.float64(19.84895429899303)]
['Kho Bưu cục Thanh Sơn', 4512, np.int64(887), np.float64(19.65868794326241)]
['Kho Hà Huy Tập', 12808, np.int64(2855), np.float64(22.290755777638978)]
['Kho Hà Tĩnh', 4651, np.int64(264), np.float64(5.6761986669533435)]
['Kho Hòa Khánh', 4979, np.int64(1458), np.float64(29.282988551918056)]
['Kho Hải Dương', 9531, np.int64(2470), np.float64(25.91543384744518)]
['Kho TGDĐ Bảo Hành', 1300, np.int64(3), np.float64(0.23076923076923078)]
['Kho Thanh Xuân', 52425, np.int64(14107), np.float64(26.90891750119218)]


In [ ]:
# Kho TGDĐ Bảo Hành
# Kho Hà Tĩnh
# Kho Kiến An
# Kho Thường Tín
# Kho Amway
# Kho Digiworld
# Kho Thái Nguyên
# Kho Việt Trì
# Kho Ninh Hòa
# Kho An Giang
# Bưu cục Dự Án Vgreen Sóng Thần
# Bưu cục Dự Án Vgreen Văn Giang

In [52]:
def plot_warehouse_vd_type(df, warehouse_name):

    vd_counts = df['VD_type'].value_counts()

    labels = list(vd_counts.index)
    counts = list(vd_counts.values)
    
    target_label = "BillWebOnline"
    
    if target_label in labels:
        idx = labels.index(target_label)
        target_count = counts.pop(idx)
        target_label_val = labels.pop(idx)
        
        labels.insert(0, target_label_val)
        counts.insert(0, target_count)
    else:
        print(f" Không tìm thấy VD_type '{target_label}' trong dữ liệu của {warehouse_name}")
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(labels, counts, color='skyblue', edgecolor='black')
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, int(yval), ha='center', va='bottom')
        
    plt.title(f'Thống kê VD_type cho kho đầu tiên là {warehouse_name}')
    plt.xlabel('VD_type')
    plt.ylabel('Số lượng bill')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

check tần suất suất hiện 

In [69]:
chunks = [] 
for chunk in pd.read_csv('bill_schedule.csv',chunksize = 500_000):
    chunk['io_time'] = pd.to_datetime(chunk['io_time']) 
    chunks.append(chunk) 

sche = pd.concat(chunks, ignore_index=True) 

In [25]:
warehouse = pd.read_csv('warehouse.csv') 
set_bc = warehouse[warehouse['Bưu Cục'] == "Y"]

In [ ]:

sche = sche.sort_values(['bill_code', 'io_time']).reset_index(drop=True) 
sche['next_wh'] = sche.groupby('bill_code')['warehouse_name'].shift(-1) 
first_rows = sche.groupby('bill_code').first().reset_index() 

first_rows['is_one_time'] = (first_rows['warehouse_name'] != first_rows['next_wh']) 
first_rows = first_rows[first_rows['warehouse_name'].isin(set(set_bc['name']))]
agg_counts = first_rows.groupby(['is_one_time', 'io_status']).size().unstack(fill_value = 0) 
agg_counts.index = agg_counts.index.map({
    True: 'Xuất hiện 1 lần', 
    False: 'Xuất hiện > 1 lần'
})

agg_counts = agg_counts.reindex(['Xuất hiện 1 lần', 'Xuất hiện > 1 lần']).fillna(0)
ax = agg_counts.plot(kind='bar', stacked=True, figsize=(10, 6), color=['blue', 'green', 'pink'][:len(agg_counts.columns)])
plt.title('Thống kê Bill theo số lần xuất hiện của kho đầu tiên và trạng thái', fontsize=14, fontweight='bold')
plt.xlabel('Tần suất xuất hiện của kho đầu tiên', fontsize=12)
plt.ylabel('Số lượng Bill', fontsize=12)
plt.xticks(rotation=0)
for c in ax.containers:
    ax.bar_label(c, label_type='center', color='white', fontweight='bold')
plt.legend(title='Trạng thái (IN/OUT)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [87]:
print(agg_counts)
print(agg_counts.shape)

io_status              IN     OUT
is_one_time                      
Xuất hiện 1 lần     11987  785071
Xuất hiện > 1 lần  469828   15791
(2, 2)


In [99]:
from enum import auto
import plotly.express as px 
dt = agg_counts.reset_index().copy() 
dt = dt.melt(id_vars = ['is_one_time'], var_name = 'status', value_name = 'count') 
fig = px.bar(dt, x = 'is_one_time', y = 'count', color = 'status',
             category_orders={'is_one_time' : ['Xuất hiện 1 lần', "Xuất hiện > 1 lần"]}, 
             hover_data = 'count', facet_col= 'is_one_time', 
             height = 500, 
             text_auto = True, 
             text = 'count') 
fig.show()


check đến kho gửi hàng 

In [19]:
origin_head = pd.read_csv('output_all_traces/origin_head.csv')
des_tail = pd.read_csv('output_all_traces/destination_tail.csv') 
bill = pd.read_csv('bill.csv', usecols=['bill_code', 'VD_type','receiving_date'])

In [88]:
oh = origin_head.copy() 
oh = oh.merge(bill, on = 'bill_code', how = 'inner') 
oh['time_o'] = pd.to_datetime(oh['time_o']) 
oh['receiving_date'] = pd.to_datetime(oh['receiving_date'])

oh['timee']= (((oh['time_o'] - oh['receiving_date']).dt.total_seconds())/60).round(0).astype("Int64")

In [89]:
o = oh['timee'].value_counts()
print(oh.shape)
fig = px.bar(o, text_auto=True) 
fig.show()

(419566, 10)


In [95]:
is_direct = oh[oh['timee'] <= 8]
print(is_direct['kho_o'].nunique(),oh['kho_o'].nunique())
print(is_direct.shape)
print(oh['timee'])

132 143
(80559, 10)
0          94
1           4
2          64
3         131
4         162
         ... 
419561      0
419562      8
419563     80
419564     15
419565     77
Name: timee, Length: 419566, dtype: Int64


In [91]:
wh = pd.read_csv('warehouse.csv') 
set_bc = wh[wh['Bưu Cục'] == "Y"]

In [96]:
a = is_direct['kho_o'].value_counts() 
fig1 = px.bar(a, text_auto = True, width=2000) 
fig1.show()
is_bc = is_direct[is_direct['kho_o'].isin(set(set_bc['name']))]
print(is_bc['kho_o'].nunique())

132


In [97]:
b = is_direct['VD_type'].value_counts()
fig2 = px.bar(b,text_auto= True)
fig2.show()

In [98]:
c = is_direct.groupby('kho_o')['VD_type'].value_counts()
c = c.reset_index(name = 'count') 
fig3= px.bar(c,x ='kho_o', y = 'count', color = "VD_type", text_auto = True, width = 5000, height = 1500)
fig3.show()

In [2]:
oh = pd.read_csv('output_all_traces/origin_head.csv') 
bill = pd.read_csv('bill.csv')

In [15]:
oh = oh.merge(bill[['bill_code', 'origin_province', 'destination_province']], on = 'bill_code', how = 'inner')
oh['pair'] = oh['kho_o'].astype(str) + '->' + oh['kho_o1a'].astype(str)
fil = oh[((oh['pair'] == 'Kho Thanh Xuân->Kho Văn Giang') & (oh['origin_province'] == 'Tp Hồ Chí Minh'))] 
print(fil)

        bill_code           kho_o               time_o status_o  \
1767    B_0007402  Kho Thanh Xuân  2026-02-25 18:03:53      OUT   
1768    B_0007417  Kho Thanh Xuân  2026-02-25 18:03:54      OUT   
3219    B_0013018  Kho Thanh Xuân  2026-02-26 18:09:37      OUT   
3266    B_0013310  Kho Thanh Xuân  2026-02-26 18:09:31      OUT   
3268    B_0013319  Kho Thanh Xuân  2026-02-26 18:09:39      OUT   
...           ...             ...                  ...      ...   
375982  B_2545093  Kho Thanh Xuân  2026-05-11 18:56:37      OUT   
375986  B_2545105  Kho Thanh Xuân  2026-05-13 18:24:18      OUT   
376011  B_2545159  Kho Thanh Xuân  2026-05-14 18:06:39      OUT   
376012  B_2545160  Kho Thanh Xuân  2026-05-14 18:06:31      OUT   
376304  B_2546528  Kho Thanh Xuân  2026-05-27 18:47:54      OUT   

              kho_o1a             time_o1a status_o1a  actual_weight  \
1767    Kho Văn Giang  2026-02-25 22:22:09         IN           0.50   
1768    Kho Văn Giang  2026-02-25 22:22:03         